In [ ]:
# # In Google Colab
!pip install cdsapi
!pip install netcdf4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.9 MB/s eta 0:00:00


In [ ]:
import os

api_key = "1c1d3369-eca8-40cf-ab21-d40c891555ab"  # paste your actual token

with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(f"url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {api_key}\n")

print("Credentials written")

# Verify it looks correct
with open(os.path.expanduser("~/.cdsapirc"), "r") as f:
    print(f.read())

Credentials written
url: https://cds.climate.copernicus.eu/api
key: 1c1d3369-eca8-40cf-ab21-d40c891555ab



In [ ]:
import cdsapi
import xarray as xr
import numpy as np
import pandas as pd

# ── Site coordinates ──────────────────────────────────────────
sites = {
    "Eureka":   {"lat": 37.117,  "lon": -117.683},
    "Mesquite": {"lat": 36.690,  "lon": -117.155},
    "Ibex":     {"lat": 35.688,  "lon": -116.362},
}

# ── Download ERA5 monthly means ───────────────────────────────
c = cdsapi.Client()

c.retrieve(
    "reanalysis-era5-single-levels-monthly-means",
    {
        "product_type": "monthly_averaged_reanalysis",
        "variable": [
            "2m_temperature",
            "2m_dewpoint_temperature",
            "total_precipitation",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "surface_solar_radiation_downwards",
        ],
        "year": [str(y) for y in range(2000, 2026)],  # adjust to your study period
        "month": [f"{m:02d}" for m in range(1, 13)],
        "time": "00:00",
        "area": [38, -118, 35, -116],  # covers all three sites: N, W, S, E
        "format": "netcdf",
    },
    "era5_death_valley.nc"
)

print("Download complete")



2026-05-29 06:07:28,868 INFO Request ID is b1b61ccf-a60a-430e-bb13-032e2b121d52
INFO:ecmwf.datastores.legacy_client:Request ID is b1b61ccf-a60a-430e-bb13-032e2b121d52
2026-05-29 06:07:29,026 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-29 06:07:46,460 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-29 06:07:54,212 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


e2382a0c279f31c32d94d42348d452cd.zip:   0%|          | 0.00/567k [00:00<?, ?B/s]

Download complete


In [ ]:
# import cdsapi
# import xarray as xr
# import numpy as np
# import pandas as pd

# # ── Extract point values for each site ───────────────────────
# ds = xr.open_dataset("era5_death_valley.nc")

# results = []

# for site, coords in sites.items():
#     # Select nearest grid point to each site
#     ds_point = ds.sel(
#         latitude=coords["lat"],
#         longitude=coords["lon"],
#         method="nearest"
#     )

#     df = ds_point.to_dataframe().reset_index()
#     df["Site"] = site

#     # Convert temperature from Kelvin to Celsius
#     if "t2m" in df.columns:
#         df["t2m_celsius"] = df["t2m"] - 273.15

#     # Convert dewpoint from Kelvin to Celsius
#     if "d2m" in df.columns:
#         df["d2m_celsius"] = df["d2m"] - 273.15

#     # Calculate relative humidity using Magnus formula
#     if "t2m" in df.columns and "d2m" in df.columns:
#         T  = df["t2m"] - 273.15
#         Td = df["d2m"] - 273.15
#         df["relative_humidity"] = 100 * (
#             np.exp((17.625 * Td) / (243.04 + Td)) /
#             np.exp((17.625 * T)  / (243.04 + T))
#         )

#     # Calculate wind speed from u and v components
#     if "u10" in df.columns and "v10" in df.columns:
#         df["wind_speed"] = np.sqrt(df["u10"]**2 + df["v10"]**2)

#     # Convert precipitation from metres to mm
#     if "tp" in df.columns:
#         df["precip_mm"] = df["tp"] * 1000

#     results.append(df)

# era5_df = pd.concat(results, ignore_index=True)

# # ── Summary table ─────────────────────────────────────────────
# summary = era5_df.groupby("Site").agg(
#     Mean_Temp_C=("t2m_celsius",   "mean"),
#     Mean_Precip_mm=("precip_mm",  "mean"),
#     Mean_Wind_ms=("wind_speed",   "mean"),
# ).round(2)

# print("\nERA5 Climate Summary by Dune Field:")
# print(summary)

# era5_df.to_csv("era5_all_sites_2000-2026.csv", index=False)
# summary.to_csv("era5_summary_2000-2026.csv")
# print("\nSaved to era5_all_sites.csv and era5_summary.csv")

ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'h5netcdf', 'scipy']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [ ]:
import os

# Check file exists and its size
size = os.path.getsize("era5_death_valley.nc")
print(f"File size: {size} bytes")

# Check the first few bytes to identify the format
with open("era5_death_valley.nc", "rb") as f:
    header = f.read(4)
print(f"File header bytes: {header}")
# NetCDF4 starts with b'\\x89HDF'
# GRIB starts with b'GRIB'
# NetCDF3 starts with b'CDF\\x01'


File size: 580322 bytes
File header bytes: b'PK\x03\x04'


In [ ]:
import zipfile
import os

# Unzip the downloaded file
with zipfile.ZipFile("era5_death_valley.nc", "r") as z:
    print("Files inside zip:", z.namelist())
    z.extractall("era5_extracted/")

# List what was extracted
for f in os.listdir("era5_extracted/"):
    print(f)

Files inside zip: ['data_stream-moda_stepType-avgua.nc', 'data_stream-moda_stepType-avgad.nc']
data_stream-moda_stepType-avgua.nc
data_stream-moda_stepType-avgad.nc


In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

# ── Open both files ───────────────────────────────────────────
ds_avg = xr.open_dataset("era5_extracted/data_stream-moda_stepType-avgua.nc")
ds_acc = xr.open_dataset("era5_extracted/data_stream-moda_stepType-avgad.nc")

print("Averaged variables:", list(ds_avg.data_vars))
print("Accumulated variables:", list(ds_acc.data_vars))
print("\nAveraged dims:", dict(ds_avg.dims))
print("Accumulated dims:", dict(ds_acc.dims))

Averaged variables: ['t2m', 'd2m', 'u10', 'v10']
Accumulated variables: ['tp', 'ssrd']

Averaged dims: {'valid_time': 312, 'latitude': 13, 'longitude': 9}
Accumulated dims: {'valid_time': 312, 'latitude': 13, 'longitude': 9}


/tmp/ipykernel_3871/2282040973.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("\nAveraged dims:", dict(ds_avg.dims))
/tmp/ipykernel_3871/2282040973.py:12: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Accumulated dims:", dict(ds_acc.dims))


In [ ]:
# ── Merge both datasets ───────────────────────────────────────
ds = xr.merge([ds_avg, ds_acc])

# ── Site coordinates ──────────────────────────────────────────
sites = {
    "Eureka":   {"lat": 37.117,  "lon": -117.683},
    "Mesquite": {"lat": 36.690,  "lon": -117.155},
    "Ibex":     {"lat": 35.688,  "lon": -116.362},
}

results = []

for site, coords in sites.items():
    ds_point = ds.sel(
        latitude=coords["lat"],
        longitude=coords["lon"],
        method="nearest"
    )

    df = ds_point.to_dataframe().reset_index()
    df["Site"] = site

    # Temperature K to C
    df["t2m_celsius"]  = df["t2m"] - 273.15
    df["d2m_celsius"]  = df["d2m"] - 273.15

    # Relative humidity — Magnus formula
    T  = df["t2m_celsius"]
    Td = df["d2m_celsius"]
    df["relative_humidity"] = 100 * (
        np.exp((17.625 * Td) / (243.04 + Td)) /
        np.exp((17.625 * T)  / (243.04 + T))
    )

    # Wind speed from components
    df["wind_speed_ms"] = np.sqrt(df["u10"]**2 + df["v10"]**2)

    # Precipitation m to mm
    df["precip_mm"] = df["tp"] * 1000

    # Solar radiation J/m2 to MJ/m2
    df["solar_MJ"] = df["ssrd"] / 1e6

    results.append(df)

era5_df = pd.concat(results, ignore_index=True)

# ── Summary table ─────────────────────────────────────────────
summary = era5_df.groupby("Site").agg(
    Mean_Temp_C        = ("t2m_celsius",      "mean"),
    Mean_Precip_mm     = ("precip_mm",        "mean"),
    Mean_Wind_ms       = ("wind_speed_ms",    "mean"),
    Mean_RH_pct        = ("relative_humidity","mean"),
    Mean_Solar_MJm2    = ("solar_MJ",         "mean"),
).round(2)

print("\nERA5 Climate Summary by Dune Field:")
print(summary)

# ── Save ──────────────────────────────────────────────────────
era5_df.to_csv("era5_all_sites_2000-2026.csv", index=False)
summary.to_csv("era5_summary_2000-2026.csv")
print("\nSaved to era5_all_sites.csv and era5_summary.csv")


ERA5 Climate Summary by Dune Field:
          Mean_Temp_C  Mean_Precip_mm  Mean_Wind_ms  Mean_RH_pct  \
Site                                                               
Eureka          15.18            0.62          0.59    29.260000   
Ibex            21.52            0.30          0.91    23.299999   
Mesquite        19.65            0.37          0.69    23.320000   

          Mean_Solar_MJm2  
Site                       
Eureka          21.110001  
Ibex            21.240000  
Mesquite        21.080000  

Saved to era5_all_sites.csv and era5_summary.csv


/tmp/ipykernel_3871/1023595965.py:2: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  ds = xr.merge([ds_avg, ds_acc])
/tmp/ipykernel_3871/1023595965.py:2: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge([ds_avg, ds_acc])
/tmp/ipykernel_3871/1023595965.py:2: FutureWarning: In a future version of xarray the default value for co

In [ ]:
# Annual means — one row per site per year
era5_df["year"]  = pd.to_datetime(era5_df["valid_time"]).dt.year
era5_df["month"] = pd.to_datetime(era5_df["valid_time"]).dt.month

annual = era5_df.groupby(["Site", "year"]).agg(
    Mean_Temp_C        = ("t2m_celsius",       "mean"),
    Total_Precip_mm    = ("precip_mm",         "sum"),
    Mean_Wind_ms       = ("wind_speed_ms",     "mean"),
    Mean_RH_pct        = ("relative_humidity", "mean"),
).round(2)

print("\nAnnual summary by site:")
print(annual.to_string())

# Seasonal means — useful for showing summer vs winter differences
era5_df["season"] = era5_df["month"].map({
    12:"Winter", 1:"Winter", 2:"Winter",
    3:"Spring",  4:"Spring",  5:"Spring",
    6:"Summer",  7:"Summer",  8:"Summer",
    9:"Fall",   10:"Fall",   11:"Fall"
})

seasonal = era5_df.groupby(["Site","season"]).agg(
    Mean_Temp_C     = ("t2m_celsius",       "mean"),
    Total_Precip_mm = ("precip_mm",         "sum"),
    Mean_RH_pct     = ("relative_humidity", "mean"),
).round(2)

print("\nSeasonal summary by site:")
print(seasonal.to_string())


Annual summary by site:
               Mean_Temp_C  Total_Precip_mm  Mean_Wind_ms  Mean_RH_pct
Site     year                                                         
Eureka   2000    15.050000             7.90          0.56    32.610001
         2001    14.050000            10.05          0.71    36.470001
         2002    15.290000             4.00          0.55    26.910000
         2003    15.290000             7.31          0.61    33.590000
         2004    14.200000            10.07          0.67    34.660000
         2005    14.810000            10.29          0.57    33.740002
         2006    14.610000             7.50          0.59    28.820000
         2007    15.640000             4.25          0.66    24.750000
         2008    14.460000             7.88          0.64    26.660000
         2009    14.430000             6.73          0.60    27.340000
         2010    13.810000            12.77          0.56    33.080002
         2011    13.940000             5.80         